### LLM

In [ ]:
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import re
import shutil
import os

import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

HF_TOKEN = ""

# Chroma 캐시 디렉토리 정의
CHROMA_DB_PATH = "./chroma_db"

# 이전 벡터 DB 삭제 (매번 새로 생성하도록)
# if os.path.exists(CHROMA_DB_PATH):
#     print(f"이전 벡터 데이터베이스 삭제: {CHROMA_DB_PATH}")
#     shutil.rmtree(CHROMA_DB_PATH)

# 1. 문서 로드 및 분할
# 'manual.pdf'라는 파일이 있다고 가정합니다.
print("문서를 읽고 분할하는 중...")
loader = PyPDFLoader(r"C:\sjbang\STUDY\Classification\data\waste_train.pdf") # 내 PDF 경로로 수정

documents = loader.load()
print(f"로드된 문서 수: {len(documents)}")

# 2. PDF 안의 텍스트를 한 줄짜리 평서문 문장으로 자동 가공하기
transformed_documents = []

for doc in documents:
    lines = doc.page_content.split('\n')
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        if "data" in line.lower() or "category" in line.lower() or "항목" in line:
            continue
            
        # 💡 [정치선 수정] 공백이 1개 이상 연속된 모든 곳을 쪼갭니다.
        # 기존 r'\s{2,}' 대신 r'\s+'를 쓰면 공백 개수에 상관없이 무조건 단어 단위로 다 쪼개져!
        tokens = [t.strip() for t in re.split(r'\s+|\t', line) if t.strip()]
        
        # 💡 이제 단어가 2개 이상이기만 하면 무조건 매칭되도록 보장!
        # 예: ['신문지', '종이류'] -> 길이는 2
        if len(tokens) >= 2:
            item = tokens[0]
            # 카테고리에 공백이 섞여 있을 수도 있으니 맨 마지막 단어를 카테고리로 지정
            category = tokens[-1] 
            
            transformed_text = f"{item}의 분리배출 카테고리는 {category}입니다."
            transformed_documents.append(Document(page_content=transformed_text))

# 🚨 디버깅용: 가공된 데이터가 진짜 있는지 안전장치 추가
print(f"🚀 최종 가공된 문장 수: {len(transformed_documents)}개")

if len(transformed_documents) == 0:
    print("❌ [경고] PDF 파싱 조건이 맞지 않아 가공된 문장이 0개입니다! 위쪽의 tokens 분리 규칙을 확인하세요.")
else:
    print(f"📋 추출 샘플 확인: {transformed_documents[0].page_content}")

# =========================================================================
# 5. 텍스트 임베딩 및 벡터 데이터베이스(Chroma) 생성 및 저장
# =========================================================================
if len(documents) == 0:
    print("경고: 문서를 찾을 수 없습니다. 파일 경로를 확인하세요.")
else:
    # 긴 문서를 모델이 읽기 편하게 적당한 크기(Chunk)로 자릅니다.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)
    print(f"분할된 텍스트 청크 수: {len(texts)}")

    # 2. 문서를 숫자로 변환 (Embedding)
    # 한국어/영어 모두 지원하는 가벼운 모델을 사용합니다.
    print("임베딩 모델 로드 중...")
    embeddings = HuggingFaceEmbeddings(
        # model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        # model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_name = "BAAI/bge-m3",
        model_kwargs={'device': 'cpu'} # GPU가 있다면 'cuda'로 변경
    )

    # 3. 벡터 데이터베이스 생성 (메모리가 아닌 디스크 기반)
    # 추출된 특징들을 ChromaDB에 저장합니다.
    print("벡터 데이터베이스 생성 중...")
    vector_db = Chroma.from_documents(
        # documents=texts,
        documents=transformed_documents,  # 👈 가공된 문장 리스트를 주입!
        embedding=embeddings,
        persist_directory=CHROMA_DB_PATH
    )
    vector_db.persist()  # 디스크에 저장
    print("벡터 데이터베이스 생성 및 저장 완료!")

    # 4. 답변 생성 모델 (LLM) 로드
    # 여기서는 아주 가벼운 'Gemma' 또는 'Qwen' 모델을 추천합니다.
    model_id = "Qwen/Qwen2.5-1.5B-Instruct" # 혹은 "Qwen/Qwen2.5-1.5B-Instruct"
    # model_id = "google/gemma-4-31B-it:together"
    # model_id = "google/gemma-4-E4B-it" # 사용권한 취득 필요
    print(f"생성 모델({model_id}) 로드 중... (시간이 좀 걸릴 수 있습니다)")

    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map={"": 0},
        # device_map="auto",
        token=HF_TOKEN
    )

    # LangChain과 연결하기 위한 파이프라인 구성
    pipe = pipeline(
        "text-generation", 
        model=model, 
        tokenizer=tokenizer, 
        max_new_tokens=128,
        temperature=0.1, # 답변의 일관성을 위해 낮게 설정
        return_full_text=False  # 생성된 텍스트만 반환 (입력 프롬프트 제외)
    )
    llm = HuggingFacePipeline(pipeline=pipe)

    # 5. 최신 방식의 RAG 체인 구축
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    # 프롬프트 템플릿 정의
    prompt = ChatPromptTemplate.from_template(

    """<|im_start|>system
    당신은 쓰레기 분리배출 카테고리를 정확하게 안내하는 챗봇입니다.
    주어진 컨텍스트 정보만을 바탕으로 질문에 답변하세요.

    [출력 규칙]
    1. 불필요한 설명, 인사말, 수식어는 전부 제외합니다.
    2. 오직 해당하는 카테고리 이름 뒤에 "~입니다" 또는 "~류입니다"만 붙여서 한 줄로 명사형 종결 어미로만 답변하세요.
    3. 만약 컨텍스트를 보고 답을 알 수 없다면 오직 "모릅니다"라고만 답변하세요.

    [답변 예시]
    질문: 신문지는 어디에 버려야 하나요?
    답변: 종이류입니다.

    질문: 콜라캔의 category는 무엇인가요?
    답변: 캔/금속류입니다.
    <|im_end|>
    <|im_start|>user
    컨텍스트:
    {context}

    질문: {question}<|im_end|>
    <|im_start|>assistant
    답변:"""
            )

    # RAG 체인 구성: retriever -> context 포매팅 -> LLM -> 출력
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    # query  = "계란껍질의 category는 무엇인가요?"
    # print(f"\n질문: {query}")
    # result = rag_chain.invoke(query)

    # # 결과에서 "Assistant:" 이후의 텍스트만 추출
    # if "Assistant:" in result:
    #     answer = result.split("Assistant:", 1)[-1].strip()
    # else:
    #     answer = result.strip()

    # print("\n=== AI 답변 ===")
    # print(answer)

c:\Users\SPACE\anaconda3\envs\LLM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0521 09:19:18.299000 17560 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


문서를 읽고 분할하는 중...
로드된 문서 수: 1
🚀 최종 가공된 문장 수: 36개
📋 추출 샘플 확인: 신문지의 분리배출 카테고리는 종이류입니다.
분할된 텍스트 청크 수: 1
임베딩 모델 로드 중...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17560\2746936792.py:87: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24500.98it/s]


벡터 데이터베이스 생성 중...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17560\2746936792.py:103: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()  # 디스크에 저장


벡터 데이터베이스 생성 및 저장 완료!
생성 모델(Qwen/Qwen2.5-1.5B-Instruct) 로드 중... (시간이 좀 걸릴 수 있습니다)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 147.56it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_17560\2746936792.py:131: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


### 코사인유사도

In [2]:
# 코사인 유사도

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 6. Retriever가 찾아온 문서의 코사인 유사도 검증
# ==========================================

print("\n🔍 === Retriever 코사인 유사도 검증 코너 ===")
query = "계란껍질의 category는 무엇인가요?"
print(f"사용자 질문: '{query}'\n")

# 1. 사용자의 질문을 임베딩 벡터로 변환
query_vector = embeddings.embed_query(query)
# 계산을 위해 2차원 배열 형태로 모양 변경 (1, Vector_Dimension)
query_vector_2d = np.array(query_vector).reshape(1, -1) 

# 2. Retriever를 통해 질문과 유사한 PDF 문서 조각 TOP 3 가져오기
retrieved_docs = retriever.invoke(query)

# 3. 가져온 문서 조각들을 하나씩 돌면서 유사도 점수 계산하기
for i, doc in enumerate(retrieved_docs, 1):
    # 문서 조각 내용을 임베딩 벡터로 변환
    doc_vector = embeddings.embed_query(doc.page_content)
    doc_vector_2d = np.array(doc_vector).reshape(1, -1)
    
    # 💡 핵심: 질문 벡터와 문서 벡터 사이의 코사인 유사도 계산!
    similarity_score = cosine_similarity(query_vector_2d, doc_vector_2d)[0][0]
    
    print(f"[{i}순위 문서 조각]")
    print(f"📊 코사인 유사도 점수: {similarity_score:.4f} (1에 가까울수록 좋음)")
    print(f"📄 가져온 텍스트 일부: {doc.page_content[:150]}...")
    print("-" * 50)


🔍 === Retriever 코사인 유사도 검증 코너 ===
사용자 질문: '계란껍질의 category는 무엇인가요?'

[1순위 문서 조각]
📊 코사인 유사도 점수: 0.7105 (1에 가까울수록 좋음)
📄 가져온 텍스트 일부: 계란껍질의 분리배출 카테고리는 일반쓰레기입니다....
--------------------------------------------------
[2순위 문서 조각]
📊 코사인 유사도 점수: 0.7105 (1에 가까울수록 좋음)
📄 가져온 텍스트 일부: 계란껍질의 분리배출 카테고리는 일반쓰레기입니다....
--------------------------------------------------
[3순위 문서 조각]
📊 코사인 유사도 점수: 0.5892 (1에 가까울수록 좋음)
📄 가져온 텍스트 일부: 양파껍질의 분리배출 카테고리는 일반쓰레기입니다....
--------------------------------------------------


In [15]:
query  = "신발의 category는 무엇인가요?"
print(f"\n질문: {query}")
result = rag_chain.invoke(query)

# 결과에서 "Assistant:" 이후의 텍스트만 추출
if "Assistant:" in result:
    answer = result.split("Assistant:", 1)[-1].strip()
else:
    answer = result.strip()

print("\n=== AI 답변 ===")
print(answer)


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



질문: 신발의 category는 무엇인가요?

=== AI 답변 ===
신발은 일반쓰레기입니다. 이는 신발 종류와 사용 기간, 그리고 소비자의 개인적인 선택에 따라 다릅니다. 그러나 일반적으로 신발은 대부분 일반쓰레기로 분류됩니다. 

신발은 주로 두 가지
